# Minimal REINFORCE Agent for Bluebird Gymnasium

This notebook is the notebook companion to `examples/minimal_reinforce_agent.py`.

It is intentionally educational rather than production-grade. It shows:

- how to configure a small Bluebird environment
- how a policy network maps observations to logits
- how stochastic action sampling works during training
- how episode rewards become discounted returns
- how loss, backpropagation, and optimizer updates fit together
- how to visualize learning progress
- how to compare learned performance against a random baseline
- how to run periodic evaluation, save checkpoints, restore the best model, and render a GIF

## Environment note

This notebook assumes you are already running the **correct Python/Jupyter kernel**:
one with `gymnasium`, `torch`, and the Bluebird project dependencies installed.

Do the package setup in your shell or project virtual environment first,
then open this notebook with that kernel. The notebook does **not** try to install
packages itself.

## Imports and path setup

This cell makes the notebook runnable from either the `bluebird-gymnasium` directory
or the repo root by adding the local package paths to `sys.path`.

In [ ]:
from __future__ import annotations

import random
import shutil
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from IPython.display import Image, display

search_roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
gym_root = None
dt_root = None

for candidate in search_roots:
    if (candidate / 'bluebird_gymnasium').exists():
        gym_root = candidate
        sibling_dt = candidate.parent / 'bluebird-dt'
        if sibling_dt.exists():
            dt_root = sibling_dt
        break
    if (candidate / 'bluebird-gymnasium').exists() and (candidate / 'bluebird-dt').exists():
        gym_root = candidate / 'bluebird-gymnasium'
        dt_root = candidate / 'bluebird-dt'
        break

if gym_root is None or dt_root is None:
    raise RuntimeError('Could not locate local bluebird-gymnasium and bluebird-dt package roots.')

sys.path.insert(0, str(gym_root))
sys.path.insert(0, str(dt_root))

from bluebird_gymnasium.envs import EnvConfig, ViewType
from bluebird_gymnasium.envs.sector_i import SectorIEnv
from bluebird_gymnasium.utils.video import generate_video

print(f'Using bluebird-gymnasium from: {gym_root}')
print(f'Using bluebird-dt from: {dt_root}')
print(f'Torch version: {torch.__version__}')

## Policy networks and agents

This notebook uses two simple agents:

- `SharedPolicyAgent`: a trainable neural-network policy
- `RandomAgent`: a fixed baseline for comparison

The policy network outputs one **logit** per action. During training we sample
from the categorical distribution defined by those logits. During evaluation we
use deterministic `argmax` action selection.

In [ ]:
class PolicyNetwork(nn.Module):
    """Neural network that maps one aircraft observation to action logits."""

    def __init__(
        self,
        observation_dimension: int,
        number_of_actions: int,
        hidden_units: int = 128,
    ) -> None:
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(observation_dimension, hidden_units),
            nn.ReLU(),
            nn.Linear(hidden_units, hidden_units),
            nn.ReLU(),
            nn.Linear(hidden_units, number_of_actions),
        )

    def forward(self, observation_batch: torch.Tensor) -> torch.Tensor:
        return self.layers(observation_batch)


class SharedPolicyAgent:
    """One shared policy network reused for every aircraft."""

    def __init__(
        self,
        observation_dimension: int,
        number_of_actions: int,
        learning_rate: float = 1e-3,
        hidden_units: int = 128,
    ) -> None:
        self.policy_network = PolicyNetwork(
            observation_dimension=observation_dimension,
            number_of_actions=number_of_actions,
            hidden_units=hidden_units,
        )
        self.optimizer = optim.Adam(
            self.policy_network.parameters(),
            lr=learning_rate,
        )

    def sample_training_actions(
        self,
        observation_by_callsign: dict[str, np.ndarray],
    ) -> tuple[dict[str, int], dict[str, torch.Tensor]]:
        action_by_callsign: dict[str, int] = {}
        log_probability_by_callsign: dict[str, torch.Tensor] = {}

        for callsign, observation_vector in observation_by_callsign.items():
            observation_tensor = torch.tensor(
                observation_vector,
                dtype=torch.float32,
            ).unsqueeze(0)
            action_logits = self.policy_network(observation_tensor)
            action_distribution = torch.distributions.Categorical(
                logits=action_logits,
            )
            sampled_action = action_distribution.sample()
            sampled_action_log_probability = action_distribution.log_prob(
                sampled_action,
            )

            action_by_callsign[callsign] = sampled_action.item()
            log_probability_by_callsign[callsign] = (
                sampled_action_log_probability.squeeze()
            )

        return action_by_callsign, log_probability_by_callsign

    def choose_evaluation_actions(
        self,
        observation_by_callsign: dict[str, np.ndarray],
    ) -> dict[str, int]:
        action_by_callsign: dict[str, int] = {}

        with torch.no_grad():
            for callsign, observation_vector in observation_by_callsign.items():
                observation_tensor = torch.tensor(
                    observation_vector,
                    dtype=torch.float32,
                ).unsqueeze(0)
                action_logits = self.policy_network(observation_tensor)
                chosen_action = torch.argmax(action_logits, dim=-1).item()
                action_by_callsign[callsign] = chosen_action

        return action_by_callsign

    def update_policy_from_episode(
        self,
        log_probability_per_step: list[torch.Tensor],
        reward_per_step: list[float],
        discount_factor_gamma: float = 0.99,
    ) -> float | None:
        if not log_probability_per_step:
            return None

        discounted_return_per_step: list[float] = []
        running_discounted_return = 0.0

        for reward in reversed(reward_per_step):
            running_discounted_return = (
                reward + discount_factor_gamma * running_discounted_return
            )
            discounted_return_per_step.append(running_discounted_return)

        discounted_return_per_step.reverse()
        returns_tensor = torch.tensor(
            discounted_return_per_step,
            dtype=torch.float32,
        )

        if returns_tensor.numel() > 1:
            returns_std = returns_tensor.std(unbiased=False)
            if returns_std > 1e-8:
                returns_tensor = (
                    (returns_tensor - returns_tensor.mean())
                    / (returns_std + 1e-8)
                )

        policy_loss_terms: list[torch.Tensor] = []
        for action_log_probability, discounted_return in zip(
            log_probability_per_step,
            returns_tensor,
        ):
            policy_loss_terms.append(-action_log_probability * discounted_return)

        loss = torch.stack(policy_loss_terms).sum()
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        return loss.item()

    def save_checkpoint(self, checkpoint_path: Path, metadata: dict | None = None) -> None:
        checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
        payload = {
            'policy_state_dict': self.policy_network.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'metadata': metadata or {},
        }
        torch.save(payload, checkpoint_path)

    def load_checkpoint(self, checkpoint_path: Path, map_location: str = 'cpu') -> dict:
        payload = torch.load(checkpoint_path, map_location=map_location)
        self.policy_network.load_state_dict(payload['policy_state_dict'])
        if 'optimizer_state_dict' in payload:
            self.optimizer.load_state_dict(payload['optimizer_state_dict'])
        return payload.get('metadata', {})


class RandomAgent:
    """Simple random baseline for comparison."""

    def __init__(self, number_of_actions: int) -> None:
        self.number_of_actions = number_of_actions

    def choose_evaluation_actions(
        self,
        observation_by_callsign: dict[str, np.ndarray],
    ) -> dict[str, int]:
        return {
            callsign: random.randrange(self.number_of_actions)
            for callsign in observation_by_callsign.keys()
        }

## Environment configuration and rollout helpers

This configuration keeps the task deliberately small:

- `SectorIEnv`
- decentralized control
- one aircraft
- `extra_minimal` state encoding
- lateral actions only

The helper functions below support:

- one training episode
- one evaluation episode
- evaluation over many seeds
- GIF rendering for a final evaluation rollout

In [ ]:
def make_sector_i_training_config() -> EnvConfig:
    config = SectorIEnv.get_default_env_config(ViewType.DECENTRALIZED)

    config.state_repr_config = {
        'encoder_cls': 'extra_minimal',
        'k_nearest_aircraft': 1,
    }

    config.action_config = {
        'simple_heading_left': True,
        'simple_heading_right': True,
        'simple_fl_climb': False,
        'simple_fl_descent': False,
        'simple_fl_exit': False,
    }

    config.reward_config = {
        'fns': [
            'position_status_const',
            'lateral_centreline_distance_shaped',
            'safety_simple_avoidance_exp',
        ],
        'coeffs': [1.0, 1.0, 1.2],
    }

    config.view_config = {
        'type': ViewType.DECENTRALIZED.value,
        'decentralized_params': {},
    }

    config.scenario_config = {
        'cls': 'tactical',
        'args': {
            'num_aircraft': 1,
            'balance': [0.0, 0.0, 1.0],
        },
    }

    return config


def run_one_training_episode(
    environment: SectorIEnv,
    agent: SharedPolicyAgent,
    random_seed: int,
    discount_factor_gamma: float,
) -> tuple[float, int, float | None]:
    random.seed(random_seed)
    np.random.seed(random_seed)
    torch.manual_seed(random_seed)

    observation_by_callsign, _info = environment.reset(seed=random_seed)

    episode_is_done = False
    episode_step_count = 0
    episode_total_reward = 0.0

    log_probability_per_step: list[torch.Tensor] = []
    reward_per_step: list[float] = []

    while not episode_is_done:
        action_by_callsign, log_probability_by_callsign = (
            agent.sample_training_actions(observation_by_callsign)
        )

        (
            next_observation_by_callsign,
            reward_by_callsign,
            done_by_callsign,
            truncated_by_callsign,
            _info,
        ) = environment.step(action_by_callsign)

        timestep_reward = (
            float(sum(reward_by_callsign.values())) if reward_by_callsign else 0.0
        )

        if log_probability_by_callsign:
            timestep_log_probability = torch.stack(
                list(log_probability_by_callsign.values())
            ).sum()
            log_probability_per_step.append(timestep_log_probability)
            reward_per_step.append(timestep_reward)

        episode_total_reward += timestep_reward
        _ = truncated_by_callsign
        episode_is_done = all(done_by_callsign.values()) if done_by_callsign else True
        observation_by_callsign = next_observation_by_callsign
        episode_step_count += 1

    loss_value = agent.update_policy_from_episode(
        log_probability_per_step=log_probability_per_step,
        reward_per_step=reward_per_step,
        discount_factor_gamma=discount_factor_gamma,
    )

    return episode_total_reward, episode_step_count, loss_value


def run_one_evaluation_episode(
    environment: SectorIEnv,
    evaluation_agent,
    random_seed: int,
) -> tuple[float, int]:
    random.seed(random_seed)
    np.random.seed(random_seed)
    torch.manual_seed(random_seed)

    observation_by_callsign, _info = environment.reset(seed=random_seed)

    episode_is_done = False
    episode_step_count = 0
    episode_total_reward = 0.0

    while not episode_is_done:
        action_by_callsign = evaluation_agent.choose_evaluation_actions(
            observation_by_callsign,
        )

        (
            next_observation_by_callsign,
            reward_by_callsign,
            done_by_callsign,
            truncated_by_callsign,
            _info,
        ) = environment.step(action_by_callsign)

        _ = truncated_by_callsign
        episode_total_reward += (
            float(sum(reward_by_callsign.values())) if reward_by_callsign else 0.0
        )
        episode_is_done = all(done_by_callsign.values()) if done_by_callsign else True
        observation_by_callsign = next_observation_by_callsign
        episode_step_count += 1

    return episode_total_reward, episode_step_count


def evaluate_agent_over_seeds(
    environment: SectorIEnv,
    evaluation_agent,
    evaluation_seeds: list[int],
) -> dict:
    rewards: list[float] = []
    steps: list[int] = []

    for random_seed in evaluation_seeds:
        total_reward, step_count = run_one_evaluation_episode(
            environment=environment,
            evaluation_agent=evaluation_agent,
            random_seed=random_seed,
        )
        rewards.append(total_reward)
        steps.append(step_count)

    return {
        'seeds': evaluation_seeds,
        'rewards': rewards,
        'steps': steps,
        'mean_reward': float(np.mean(rewards)),
        'std_reward': float(np.std(rewards)),
        'mean_steps': float(np.mean(steps)),
    }


def render_evaluation_rollout_to_gif(
    agent: SharedPolicyAgent,
    random_seed: int,
    render_dir: Path,
    gif_name: str = 'trained_policy_eval',
) -> Path:
    render_config = make_sector_i_training_config()
    render_config.radar_config['display_actions'] = True
    render_config.radar_config['render_dir'] = str(render_dir)
    render_config.radar_config['prefix'] = 'frame'

    if render_dir.exists():
        shutil.rmtree(render_dir)
    render_dir.mkdir(parents=True, exist_ok=True)

    render_environment = SectorIEnv(config=render_config)
    render_environment.set_render_mode('file')

    random.seed(random_seed)
    np.random.seed(random_seed)
    torch.manual_seed(random_seed)

    observation_by_callsign, _info = render_environment.reset(seed=random_seed)

    # In this Bluebird branch, file-mode rendering is not triggered
    # automatically by reset() or step(). Call render() explicitly.
    render_environment.render()

    episode_is_done = False

    while not episode_is_done:
        action_by_callsign = agent.choose_evaluation_actions(observation_by_callsign)
        (
            next_observation_by_callsign,
            reward_by_callsign,
            done_by_callsign,
            truncated_by_callsign,
            _info,
        ) = render_environment.step(action_by_callsign)
        _ = reward_by_callsign, truncated_by_callsign

        # Save one frame after every evaluation step.
        render_environment.render()

        episode_is_done = all(done_by_callsign.values()) if done_by_callsign else True
        observation_by_callsign = next_observation_by_callsign

    png_frames = sorted(render_dir.glob(f"{render_config.radar_config['prefix']}_*.png"))
    if not png_frames:
        raise RuntimeError(
            f'No rendered PNG frames were written to {render_dir}. '
            'Expected at least one frame before GIF generation.'
        )

    generate_video(
        render_dir=str(render_dir),
        frame_prefix=render_config.radar_config['prefix'],
        video_filename=gif_name,
        clean_up=False,
    )
    render_environment.close()
    return render_dir / f'{gif_name}.gif'


## Set up the environment and inspect the shapes

For this notebook, the most important values are:

- `observation_dimension`: how many numbers are in one aircraft observation vector
- `number_of_actions`: how many discrete actions the policy can choose from

In [ ]:
config = make_sector_i_training_config()
environment = SectorIEnv(config=config)

observation_dimension = environment.observation_space.shape[0]
number_of_actions = environment.action_space.n

print(
    'environment shapes:',
    f'observation_dimension={observation_dimension}',
    f'number_of_actions={number_of_actions}',
)

## Hyperparameters and experiment settings

This version of the notebook adds:

- periodic evaluation during training
- a larger held-out evaluation set
- checkpoint saving
- automatic tracking of the best evaluated model
- a random-policy baseline

In [ ]:
learning_rate = 1e-3
hidden_units = 128
discount_factor_gamma = 0.99
number_of_training_episodes = 100
training_seed_start = 100
periodic_eval_interval = 10
heldout_evaluation_seeds = list(range(200, 220))
checkpoint_dir = Path.cwd() / 'checkpoints' / 'minimal_reinforce_agent'
latest_checkpoint_path = checkpoint_dir / 'latest.pt'
best_checkpoint_path = checkpoint_dir / 'best.pt'

agent = SharedPolicyAgent(
    observation_dimension=observation_dimension,
    number_of_actions=number_of_actions,
    learning_rate=learning_rate,
    hidden_units=hidden_units,
)
random_agent = RandomAgent(number_of_actions=number_of_actions)

training_rewards: list[float] = []
training_steps: list[int] = []
training_losses: list[float] = []

periodic_eval_episodes: list[int] = []
periodic_eval_learned_mean_rewards: list[float] = []
periodic_eval_learned_std_rewards: list[float] = []
periodic_eval_random_mean_rewards: list[float] = []
periodic_eval_random_std_rewards: list[float] = []
periodic_eval_learned_mean_steps: list[float] = []
periodic_eval_random_mean_steps: list[float] = []

best_mean_evaluation_reward = float('-inf')
best_checkpoint_metadata: dict = {}

## Training loop with periodic evaluation and checkpointing

Every `periodic_eval_interval` episodes, the notebook:

- evaluates the current learned policy on the held-out evaluation seeds
- evaluates a random baseline on the same seeds
- saves a `latest.pt` checkpoint
- overwrites `best.pt` if the learned policy achieves a new best mean evaluation reward

In [ ]:
for episode_index in range(number_of_training_episodes):
    random_seed = training_seed_start + episode_index
    total_reward, step_count, loss_value = run_one_training_episode(
        environment=environment,
        agent=agent,
        random_seed=random_seed,
        discount_factor_gamma=discount_factor_gamma,
    )

    training_rewards.append(total_reward)
    training_steps.append(step_count)
    training_losses.append(float('nan') if loss_value is None else loss_value)

    print(
        '[train]',
        f'episode={episode_index:03d}',
        f'seed={random_seed}',
        f'reward={total_reward:.3f}',
        f'steps={step_count}',
        f'loss={loss_value}',
    )

    should_run_periodic_eval = (
        (episode_index + 1) % periodic_eval_interval == 0
        or episode_index == number_of_training_episodes - 1
    )

    if should_run_periodic_eval:
        learned_eval = evaluate_agent_over_seeds(
            environment=environment,
            evaluation_agent=agent,
            evaluation_seeds=heldout_evaluation_seeds,
        )
        random_eval = evaluate_agent_over_seeds(
            environment=environment,
            evaluation_agent=random_agent,
            evaluation_seeds=heldout_evaluation_seeds,
        )

        periodic_eval_episodes.append(episode_index + 1)
        periodic_eval_learned_mean_rewards.append(learned_eval['mean_reward'])
        periodic_eval_learned_std_rewards.append(learned_eval['std_reward'])
        periodic_eval_random_mean_rewards.append(random_eval['mean_reward'])
        periodic_eval_random_std_rewards.append(random_eval['std_reward'])
        periodic_eval_learned_mean_steps.append(learned_eval['mean_steps'])
        periodic_eval_random_mean_steps.append(random_eval['mean_steps'])

        metadata = {
            'episode': episode_index + 1,
            'train_seed': random_seed,
            'learned_mean_reward': learned_eval['mean_reward'],
            'learned_std_reward': learned_eval['std_reward'],
            'random_mean_reward': random_eval['mean_reward'],
            'random_std_reward': random_eval['std_reward'],
            'evaluation_seeds': heldout_evaluation_seeds,
        }
        agent.save_checkpoint(latest_checkpoint_path, metadata=metadata)

        if learned_eval['mean_reward'] > best_mean_evaluation_reward:
            best_mean_evaluation_reward = learned_eval['mean_reward']
            best_checkpoint_metadata = metadata
            agent.save_checkpoint(best_checkpoint_path, metadata=metadata)
            checkpoint_note = 'new best checkpoint'
        else:
            checkpoint_note = 'latest checkpoint only'

        print(
            '[periodic-eval]',
            f'episode={episode_index + 1:03d}',
            f'learned_mean_reward={learned_eval["mean_reward"]:.3f}',
            f'learned_std_reward={learned_eval["std_reward"]:.3f}',
            f'random_mean_reward={random_eval["mean_reward"]:.3f}',
            f'random_std_reward={random_eval["std_reward"]:.3f}',
            checkpoint_note,
        )

## Restore the best evaluated model

The training loop may end on a policy that is not the best one seen so far.
This cell reloads the checkpoint with the highest held-out mean evaluation reward.

In [ ]:
if not best_checkpoint_path.exists():
    raise FileNotFoundError(f'Best checkpoint not found: {best_checkpoint_path}')

loaded_metadata = agent.load_checkpoint(best_checkpoint_path)
print('Reloaded best checkpoint from:', best_checkpoint_path)
print('Best checkpoint metadata:')
loaded_metadata

## Final evaluation of the best checkpoint vs random baseline

This uses the larger held-out evaluation set and compares:

- the best learned policy checkpoint
- a random baseline on the same seeds

In [ ]:
best_policy_eval = evaluate_agent_over_seeds(
    environment=environment,
    evaluation_agent=agent,
    evaluation_seeds=heldout_evaluation_seeds,
)
random_policy_eval = evaluate_agent_over_seeds(
    environment=environment,
    evaluation_agent=random_agent,
    evaluation_seeds=heldout_evaluation_seeds,
)

print('Best policy evaluation mean reward:', best_policy_eval['mean_reward'])
print('Best policy evaluation std reward:', best_policy_eval['std_reward'])
print('Random baseline mean reward:', random_policy_eval['mean_reward'])
print('Random baseline std reward:', random_policy_eval['std_reward'])

## Useful plots

These are the most useful quick-look plots for this richer experiment setup:

- **Training reward**: raw reward and moving average during training
- **Episode length**: how long training episodes run
- **Periodic evaluation**: learned policy vs random baseline over training
- **Final evaluation by seed**: best learned checkpoint vs random on the held-out seeds

In [ ]:
def moving_average(values: list[float], window: int) -> np.ndarray:
    if len(values) < window:
        return np.array([])
    kernel = np.ones(window) / window
    return np.convolve(np.asarray(values, dtype=float), kernel, mode='valid')

plot_window = min(10, len(training_rewards))
smoothed_rewards = moving_average(training_rewards, plot_window)
training_episode_indices = np.arange(1, len(training_rewards) + 1)
heldout_seed_indices = np.arange(len(heldout_evaluation_seeds))

fig, axes = plt.subplots(2, 2, figsize=(15, 11))

axes[0, 0].plot(training_episode_indices, training_rewards, marker='o', alpha=0.25, label='raw reward')
if len(smoothed_rewards) > 0:
    axes[0, 0].plot(
        np.arange(plot_window, len(training_rewards) + 1),
        smoothed_rewards,
        linewidth=2.5,
        color='tab:blue',
        label=f'moving average (window={plot_window})',
    )
axes[0, 0].set_title('Training Reward per Episode')
axes[0, 0].set_xlabel('Episode')
axes[0, 0].set_ylabel('Total Reward')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

axes[0, 1].plot(training_episode_indices, training_steps, marker='o', color='tab:orange')
axes[0, 1].set_title('Training Episode Length')
axes[0, 1].set_xlabel('Episode')
axes[0, 1].set_ylabel('Steps')
axes[0, 1].grid(alpha=0.3)

periodic_eval_episodes_arr = np.asarray(periodic_eval_episodes)
learned_mean_arr = np.asarray(periodic_eval_learned_mean_rewards)
learned_std_arr = np.asarray(periodic_eval_learned_std_rewards)
random_mean_arr = np.asarray(periodic_eval_random_mean_rewards)
random_std_arr = np.asarray(periodic_eval_random_std_rewards)

axes[1, 0].plot(periodic_eval_episodes_arr, learned_mean_arr, marker='o', label='learned policy')
axes[1, 0].fill_between(
    periodic_eval_episodes_arr,
    learned_mean_arr - learned_std_arr,
    learned_mean_arr + learned_std_arr,
    alpha=0.2,
)
axes[1, 0].plot(periodic_eval_episodes_arr, random_mean_arr, marker='s', label='random baseline')
axes[1, 0].fill_between(
    periodic_eval_episodes_arr,
    random_mean_arr - random_std_arr,
    random_mean_arr + random_std_arr,
    alpha=0.2,
)
axes[1, 0].set_title('Periodic Evaluation: Learned vs Random')
axes[1, 0].set_xlabel('Training Episode')
axes[1, 0].set_ylabel('Mean Evaluation Reward')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

axes[1, 1].plot(
    heldout_seed_indices,
    best_policy_eval['rewards'],
    marker='o',
    linewidth=2,
    label='best learned checkpoint',
)
axes[1, 1].plot(
    heldout_seed_indices,
    random_policy_eval['rewards'],
    marker='s',
    linewidth=2,
    label='random baseline',
)
axes[1, 1].set_xticks(heldout_seed_indices)
axes[1, 1].set_xticklabels([str(seed) for seed in heldout_evaluation_seeds], rotation=45)
axes[1, 1].set_title('Final Evaluation Reward by Seed')
axes[1, 1].set_xlabel('Held-out Evaluation Seed')
axes[1, 1].set_ylabel('Total Reward')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

fig.suptitle('Minimal REINFORCE Training Summary with Baseline and Checkpoints', fontsize=16)
fig.tight_layout()
plt.show()

print(f'Best checkpoint mean evaluation reward: {best_policy_eval["mean_reward"]:.3f}')
print(f'Random baseline mean evaluation reward: {random_policy_eval["mean_reward"]:.3f}')
print(f'Latest checkpoint path: {latest_checkpoint_path}')
print(f'Best checkpoint path: {best_checkpoint_path}')

## Render the best checkpoint and save a GIF

This section runs the **best restored checkpoint** in evaluation mode with Bluebird radar rendering enabled.
It saves individual frames to disk and then combines them into a GIF.

Notes:

- `display_actions=True` overlays actions on the radar frames
- the GIF path is printed and displayed in the notebook
- by default this uses the first held-out evaluation seed

In [ ]:
gif_seed = heldout_evaluation_seeds[0]
render_dir = Path.cwd() / 'renders' / 'minimal_reinforce_agent_eval'
gif_path = render_evaluation_rollout_to_gif(
    agent=agent,
    random_seed=gif_seed,
    render_dir=render_dir,
    gif_name=f'best_checkpoint_eval_seed_{gif_seed}',
)

print(f'Saved GIF to: {gif_path}')
display(Image(filename=str(gif_path)))

## Optional cleanup

Close the main environment if you are done with the notebook session.

In [ ]:
environment.close()